# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kabin-ux/fly-rank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Provisional Lane:** **CTR & Engagement Opportunity Scoring** (from `docs/ml-intern-dataset-and-lane-guide.md`).

**Why this lane:** Content optimization often fails because teams treat every declining page equally, missing the nuances of user engagement. By focusing on click-through rates (CTR) and engagement shifts relative to search impressions, this lane targets pages that have high visibility but are underperforming in conversion or click yield. It provides a direct operational handle for content refresh queues without requiring speculative causal claims about Google's core ranking algorithm.

In [24]:
import os
import pandas as pd

# If running in Google Colab and the data file isn't present, download/clone the repo or fetch the file
csv_path = "data/raw/content_refresh_anonymized.csv"

if not os.path.exists(csv_path):
    # Try checking alternative relative paths if cloned inside work/notebooks
    if os.path.exists("../../data/raw/content_refresh_anonymized.csv"):
        csv_path = "../../data/raw/content_refresh_anonymized.csv"
    else:
        # If we are in Colab and missing the repo structure, clone it or fetch the raw CSV
        import urllib.request
        os.makedirs("data/raw", exist_ok=True)
        url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
        try:
            urllib.request.urlretrieve(url, csv_path)
        except Exception:
            pass

df = pd.read_csv(csv_path)

total_rows = len(df)
down_count = (df['trend_direction'].str.lower() == 'down').sum()
trend_down_pct = (down_count / total_rows) * 100
mean_imps = df['impressions'].mean() if 'impressions' in df.columns else df['impressions_90d'].mean()

print(f"Dataset Shape: {total_rows} rows, {df.shape[1]} columns")
print(f"1. Total Pages: {total_rows}")
print(f"2. Downward Trend Rate: {trend_down_pct:.1f}% ({count if 'count' in locals() else down_count} pages)")
print(f"3. Mean Impressions: {mean_imps:.1f}")

Dataset Shape: 30000 rows, 44 columns
1. Total Pages: 30000
2. Downward Trend Rate: 54.2% (16262 pages)
3. Mean Impressions: 5200.4


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Search Question: Which pages with stable or high search impressions are experiencing suppressed CTR or downward trend trajectories, and which ones will recover fastest if refreshed?

Decision Supported: Deciding which subset of content to prioritize for an immediate manual content refresh vs. leaving pages untouched.

Action Taken: Content editors review the top-scored queue and rewrite metadata, expand sections, or update user-intent alignment for flagged URLs.

Cost of a Wrong Call: A false positive (recommending a refresh on a page that was naturally stabilizing or didn't need changes) wastes limited editorial hours. A false negative (missing a declining page) leads to compounding organic traffic loss before intervention.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check distribution of the target trend direction in the dataset
print(df['trend_direction'].value_counts(dropna=False))


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Analyzing the starter dataset (content_refresh_anonymized.csv) reveals clear numerical justification for this lane:Scale: The dataset contains 30,000 pseudonymized pages across multiple client domains, providing sufficient sample depth for pattern discovery.  Target Base Rate: Approximately {trend_down_pct:.1f}% of pages exhibit a downward trend direction (trend_direction == 'down'), showing that decline is common enough to build a predictive rule against without extreme class imbalance.Impression Spread: Mean impressions sit at roughly {mean_imps:.1f}, with substantial variance that isolates high-impact traffic nodes from long-tail noise.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd

# If running in Google Colab and the data file isn't present, download/clone the repo or fetch the file
csv_path = "data/raw/content_refresh_anonymized.csv"

if not os.path.exists(csv_path):
    # Try checking alternative relative paths if cloned inside work/notebooks
    if os.path.exists("../../data/raw/content_refresh_anonymized.csv"):
        csv_path = "../../data/raw/content_refresh_anonymized.csv"
    else:
        # If we are in Colab and missing the repo structure, clone it or fetch the raw CSV
        import urllib.request
        os.makedirs("data/raw", exist_ok=True)
        url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
        try:
            urllib.request.urlretrieve(url, csv_path)
        except Exception:
            pass

df = pd.read_csv(csv_path)

total_rows = len(df)
down_count = (df['trend_direction'] == 'down').sum()
trend_down_pct = (down_count / total_rows) * 100
mean_imps = df['impressions_90d'].mean()

print(f"1. Total Pages: {total_rows}")
print(f"2. Downward Trend Rate: {trend_down_pct:.1f}% ({down_count} pages)")
print(f"3. Mean Impressions: {mean_imps:.1f}")


1. Total Pages: 30000
2. Downward Trend Rate: 54.2% (16262 pages)
3. Mean Impressions: 5200.4


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What this work CAN claim: Our model identifies observed statistical associations between page features (such as historical CTR, position shifts, and content age) and subsequent downward traffic trajectories. It provides decision-support rankings to streamline editorial triage.

What this work CANNOT claim: We are not predicting Google's algorithm or proving a direct causal link. A recommendation means a page shares historical characteristics with previously refreshed pages that recovered; it does not guarantee that a specific edit will force a ranking increase.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quick verification check on feature columns available for modeling
print("Available feature columns sample:", list(df.columns[:8]))


Available feature columns sample: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.